# Lab03G — Mudança temporal agregada e calibração após o terceiro parecer

**Execute tudo, em CPU, e salve uma cópia do notebook.** Este é o único notebook novo; não repetir F/E/transferências. A execução completa faz as sensibilidades observadas e 1.000 réplicas em cada um de seis mecanismos, cada uma com 4.999 sorteios. Há mensagens de avanço a cada 50 réplicas e salvamento ao terminar cada mecanismo. É a célula mais demorada; não requer GPU.

Reutiliza o cache X autenticado do Drive, sem baixar cotações. Preserva 80/117 ativos, classes, dez setores, retornos logarítmicos, pares e ausências. Não seleciona ativos anualmente nem cria pesos de empresas. O cálculo local completo já foi realizado; a conferência desta execução será registrada separadamente.

O estudo mede mudança média/mediana entre quinquênios e inferência agregada defasada/semanal. A calibração também preserva falhas de cobertura. Não conclui homologação econômica ou uma contribuição de IA.


In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import urlretrieve
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
try:
    from IPython.display import display,Image
except ImportError:
    def display(value):print(str(value)[:6000])
    def Image(filename):
        from PIL import Image as PillowImage
        im=PillowImage.open(filename);im.load();return filename
LOCAL_MODE=bool(os.environ.get('ALEM_BETA_LOCAL'))
os.environ['ALEM_BETA_EXECUTION_ENV']='local_notebook_integration' if LOCAL_MODE else 'Colab'
BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03g')).resolve();BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN=BASE/('run_'+STAMP);RUN.mkdir();ROOT=RUN/'runtime';ROOT.mkdir()
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def script(name,*args):subprocess.run([sys.executable,str(ROOT/'scripts'/name),*map(str,args)],check=True)
print('Lab03G | CPU |', 'integração local, não Colab do autor' if LOCAL_MODE else 'Colab', '|',RUN.name)


## Fontes e entradas fixadas

Código e referências compactas vêm de um commit público imutável e são autenticados individualmente. O ZIP de retornos continua externo no Drive. Nenhum download de Yahoo/B3 é executado.


In [ ]:
SOURCE_COMMIT='880de4fa25a6ced45b137bd9d5f5126abf869f25'
SOURCE_PINS={'data/lab02/semantica_caixa_v1/recuperacao_fras_visivel_2024_2025.csv': '9888362d6022a1f92340ab204d534bc1a332df17146323a8945d363d2bef3682', 'data/lab03c/validacao_colab_20261023.json': '9c1f5c806b34d2153ae593b03c317f04f6e2058a6dafbe931713a06cba569878', 'data/lab03d/adendo_classes_equivalencia_v1.json': '019f36cb690f0b50e27e9a6be16fd1347ab55a1e4508803c160ba49c4aaa2ca5', 'data/lab03d/adendo_envelope_v1.json': 'd6d1df31e9fec714f44abeec66f61ad75117b17a09cc2374ac73e311b7f99777', 'data/lab03d/adendo_guard_causal_v1.json': '68d23baf9ba304bb54a3bc59639c68df72bc67bbcc52258f8dfa293c6902613f', 'data/lab03d/adendo_temporal_lfc_v1.json': 'd3f1a9dc3169d47b3f8f2840b50cccf1ae2de2033f5f48d95edca8a85a1073be', 'data/lab03d/inputs_manifest.json': '31225c9179356dc027a9fce2c86c4de533649ca3eac4016cf7d6832f8f5d7e5a', 'data/lab03d/plano_v1.json': 'f60b8422654df4cb27eccf69bacde50be23289cab4116a0396011ebf7daf0891', 'data/lab03d/validacao_colab_20261023.json': '6d7f60a1e56087e8a0ebbc329d3a4948028be768359433313d5b194252407fa7', 'data/lab03e/inputs_manifest.json': '052617b050e5da19a922d4c0c036f342219936dddeb1348d7d92ca4666b77a5e', 'data/lab03e/plano_v1.json': 'b8aa04548847dfa66b952e043d533233bfc8aa0c69350346cc9e145ccf3c0555', 'data/lab03e/primary/fras_cash.xlsx': '3597c6058a8358bd70c5a416924fe804cebc77feb659afaa34288053f6794df3', 'data/lab03e/primary/fras_cash_page.html': 'abfab5b634460f87c3688d23c571f77bad9a128932379aab37f4a120ca59f510', 'data/lab03e/primary/fras_extraction_audit.json': 'f8895cd3d04055417b1ad0a0fe7361621d5f1100075a56c04d1791eccb8e78da', 'data/lab03e/primary/fras_published_cash_inventory.csv': '90724375f4e8fa47fe91642186a80d48ba823db978920d72a435f3236696fbc3', 'data/lab03e/primary/retrieval.json': '3efd871a1073f92a734d831b302dbff05393f50e7a9567d850b057af29e8b3fc', 'data/lab03e/primary_sources_manifest.json': '81966a21513e79b3bd7e26edbf4e01d2ec778cdb916b24d1eb3aa66120b8c21f', 'data/lab03f/freeze.json': '7b69a88fac384f9a0d4515c0093407966eadec5abf848250c6eafa8bbe6d0388', 'data/lab03f/plano_pos_parecer_v1.json': 'f94cac62a6aa03249fb102ae7f74d5f45f3c28ca4968e503899f11b07cff6d1f', 'data/lab03g/inputs_manifest.json': '9f9fc53767bf3a95ab9fd562744cbb19466f8db65fbea7298cb7f63d230bfe15', 'data/lab03g/plano_v1.json': 'd347bfbbcef27e96aa8e8db392cb72b172a09d07140e4a3da90753fd76419da1', 'notebooks/Lab03D_inferencia_previsao.ipynb': '8ee5147c232efd43cdf1299d1059b5e8d43dc0a673b33381ab33e736b177a43f', 'results/tables/lab03b_colab_20261023/asset_betas.csv.gz': '0c2c54fc1f212d914cf083300757e2706a94d32ae762e64b8814e1a837192275', 'results/tables/lab03e_colab_20261023/asset_nuisance.csv.gz': 'ba398bf2a7ba282f3dabbf7f63c41c5379a8105de2bffc91788f25f3dbd86040', 'results/tables/lab03f_pos_parecer_20261024/aggregate.csv.gz': 'e4f1a4ff05ffc17194e826cd412eb651fe159a3055845cdcc62d7e08569e001a', 'results/tables/lab03f_pos_parecer_20261024/bootstrap_targets.npz': 'c9447e814c9aafc732a029978379f14a837ce950dd3b6ca5e04c12d19e55c1c1', 'results/tables/lab03f_pos_parecer_20261024/individual.csv.gz': 'd1e89113e8d8850563dd14713d67cb8cf026bcc78e1b70f517a70190e7815151', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_cases.csv': '35e7263855f1c8d522e5d96ea4de3264a82f368843f6d16ba46149423f55dfc6', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_families.csv': 'a980077971a21590ed6a04eacc2140bf1487390a793ab8134296c303fb8aeb86', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_replications.csv.gz': 'cb6537858edd89e4517c3bf41e1e08d0dbb410c08f10ecdc0adb1e33594f9a61', 'results/tables/lab03g_pos_parecer_v3_20261024/observed_aggregate.csv.gz': 'c1b448523e4c0e86fc1d0aba3f2dbebbb4defa2878d066c720a669ad4f3a7a53', 'results/tables/lab03g_pos_parecer_v3_20261024/observed_draws.npz': '6fadf6e088965808bc4fc39eb470bd96def884c3d0128597141ce37d304e3702', 'results/tables/lab03g_pos_parecer_v3_20261024/temporal_asset_points.csv': '84cf4fe6196ca1d2be8a955bde2a356c1c96f80fedc7b0198763c86948a135ff', 'scripts/build_lab03e_notebook.py': 'f7cc2dd6861ee77ef3221a22e6545ec25dcde1a6fbd7f0e4abac512a1e80d16d', 'scripts/build_lab03e_runtime.py': 'd8ff2d9de46d84b2fdaddfd26f9849a54ac2949dc28aa2676531eae330feb86a', 'scripts/export_lab03d.py': '413ec36ee2f63415a76489706a2f7de482410c81263677a3c71c61c2d598a701', 'scripts/export_lab03e.py': '55f88b6a8a83379c24daf9d7b42d0b49f2bb2219b2a61e46faa438131144e1f9', 'scripts/extract_lab03e_fras.py': '7107ebcedb172a1a734ef0b36a4a4bae207bc815c9bf0c45d27f422c9e81175d', 'scripts/plot_lab03d.py': 'd7d6c63adabf7ff963bec34e7b65a57fc8bc34523a13c8c0ead70bfe865ad6ec', 'scripts/plot_lab03e.py': 'db0dac0930bb8bcc1bb3e3497bd6db1be95615c8a8bd0471cd813878bfae8f43', 'scripts/plot_lab03g.py': '61b12abf72d2f7e7ebf55bafe4fb8a505b9812fdee432dad50f5265905c6b697', 'scripts/run_lab03d.py': 'aa689b2b6ff203b77daca85b09923804a719c0d08cc6b4a29b5e95a1145b9975', 'scripts/run_lab03d_lfc.py': '5110d5db922e1f1ad46867aeb5d123aa9af58bf97d9dc7d04341d2704d770151', 'scripts/run_lab03d_rank.py': '3e100649f26e9c21e5ed7dd9426fce4dd3c2184d735db2e96c6126d8fb06cd4d', 'scripts/run_lab03e_economic.py': '3ec83455d75523d854813afc908bc59b6658c1d9bd984350c13abdf1982b075a', 'scripts/run_lab03e_inference.py': 'ff3c9aa2f870913330592ba85c1e1dfca974bae3f15e52fb7dc671c1033b2045', 'scripts/run_lab03e_prediction.py': '8aa01a0df2d2e9061cd33ac7b4990551b9648f33d0fcbe392a116701f1b5018f', 'scripts/run_lab03g.py': '44251bc1b8f248b2c7a471d466df2f7529e60e9ee952dcccf8d0608b1c9094f1', 'scripts/validate_lab03d_diagnostics.py': '659bee564d506b2118d9a388a6b48b1d8461716dafdb127c099ba9a443df0f32', 'scripts/validate_lab03d_outputs.py': 'd21eb2f7a57f993b9b1a9ea38b8ebe42cdcd3f075b5fc6fd185bf449d5f96d26', 'scripts/validate_lab03d_rank.py': 'b4c178dc13b51dd72c3e4174255f58b7bb453f9a1ccced1284bb615754f559d4', 'scripts/verify_lab03e_economic.py': '0c51cf44ab70e2bdcacb6fe215b1eebbce2d6a6e290796cf2c1244f4a481fa34', 'scripts/verify_lab03e_inference.py': '211f9b9be0aa693bf1ae3c965e02e054700051daed3f7b694e14482ca002ac4d', 'scripts/verify_lab03e_prediction.py': 'ef39b6b3af27a9bd11d23ae60e3a5b0b54a857f05cf5cccb351452272d686ad2', 'scripts/verify_lab03g.py': '6ee9d1f04570f5292c920f8d67c17d71cf6a728a2d165019a4a973b135d8595b', 'src/lab02.py': 'f83bef7d5da251fb2571699572f202ee16a8b007522f3e5f1f6f1385f8540b0f', 'src/lab03c.py': '715b9cd9cec7d41a54d129fde497c14a68013a66e80a337317d91c8a2c4b20dd', 'src/lab03d_envelope.py': 'c24eab65913eb9dc3ccd76c7db962820aedc6239cb75f696c80f6755bc53db4a', 'src/lab03d_inference.py': '0d7a891a94c98539168a382bbab2ae7f16aeebf338398a2c6e98ef9498077708', 'src/lab03d_prediction.py': 'e52cad9acfefce1c27f8f8b9e3d479b0d942cd036946173ef7381fa39c05088e', 'src/lab03d_rank.py': '815799908d43bf3b0bc2c71d01750d619ae8522e050b82e426d4aee9239faa53', 'src/lab03e_common.py': '179dc3a15289f83d4780cbed88dfd07909e813f5511f87531626626419be0d96', 'src/lab03e_economic.py': '196875220ee4a64f56d023ed1df121a8cb5d0089788cfd4a64f658d6ec49dafd', 'src/lab03e_inference.py': '73ee08c05a1e0fa14cdba9f591db90e99ae877050d2dfd8622c5bf3fa7c0e517', 'src/lab03e_prediction.py': 'b6010fd76bf51c822635a994c787994c9c5f8221236d2137975096ed18fbec06', 'src/lab03f.py': 'e65dc527403d5a444b7b52caa5bfd0bde341a48a7b3b57979990c422478c68b7', 'src/lab03g.py': 'e89d3723c71da6accecd469209ce39d1e902cd0e620b83a8ac28103ac11e8c9e', 'tests/test_lab03d.py': '3639cb28cf554812a10c8d4e6ca04f6baf298be19ba5a6521d30827bb03b8b92', 'tests/test_lab03e.py': 'f72b6901926edd80c8815b79b32c6daedd9210c062db199b7232b20d01c72d5f', 'tests/test_lab03g.py': 'eb402449b1ebed3eb01acf8184439973bff1d00ea8ce149d060d8e58a03e613c'}
for name,expected in SOURCE_PINS.items():
    target=ROOT/name;target.parent.mkdir(parents=True,exist_ok=True)
    if LOCAL_MODE:shutil.copy2(Path(os.environ['ALEM_BETA_LOCAL_SOURCE'])/name,target)
    else:urlretrieve('https://raw.githubusercontent.com/robsondf/alem-do-beta-gemini/'+SOURCE_COMMIT+'/'+name,target)
    assert digest(target)==expected,'Fonte divergente: '+name
print('Fontes autenticadas:',len(SOURCE_PINS),'| commit',SOURCE_COMMIT)
X_ID='1aExT7v-GD8s5xKzM0K-0LIIJER8E8Twg'
X_SHA='297da2afde534082c398324f308df7af134c9692e7d61d6e0b2b820341ea5214'
if LOCAL_MODE:
    X_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_X'])
else:
    from google.colab import auth,drive
    auth.authenticate_user();drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False)
    CACHE=Path('/content/drive/MyDrive/alem_do_beta/lab03f_cache');CACHE.mkdir(parents=True,exist_ok=True)
    X_ARCHIVE=CACHE/(X_ID+'.zip')
    if not X_ARCHIVE.exists() or digest(X_ARCHIVE)!=X_SHA:
        name=service.files().get(fileId=X_ID,fields='name').execute()['name']
        found=None
        for candidate in Path('/content/drive/MyDrive').rglob(name):
            if candidate.is_file() and digest(candidate)==X_SHA:found=candidate;break
        if found is not None:X_ARCHIVE=found
        else:
            part=X_ARCHIVE.with_suffix('.part')
            with part.open('wb') as stream:
                download=MediaIoBaseDownload(stream,service.files().get_media(fileId=X_ID));done=False
                while not done:_,done=download.next_chunk()
            assert digest(part)==X_SHA;part.replace(X_ARCHIVE)
assert digest(X_ARCHIVE)==X_SHA
print('Cache X autenticado:',X_ARCHIVE.name,'|',X_ARCHIVE.stat().st_size,'bytes')


## Plano, testes e processamento

Plano pós-revisão, escrito antes dos resultados G, com F conhecido. A família nova de quatro testes cobre média/mediana inicial e mudança pareada; a família de 18 cobre três modelos, três painéis e dois resumos. Famílias individuais originais 197/474 preservadas. Bloco principal 21 sessões ou 4 semanas; sensibilidades completas mantidas.

As defasagens são construídas no calendário original antes da reamostragem. A calibração de tamanho/cobertura usa as famílias agregadas de dez e quatro testes, bloco 21, bancos/mercado fixados; não certifica os modelos dinâmicos/semanais. Nenhum sorteio inválido é descartado. O resultado da calibração não determina um novo método escolhido por significância.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p','test_lab03g.py','-v'],check=True)
OUTPUT=RUN/'outputs'
if LOCAL_MODE and os.environ.get('ALEM_BETA_LOCAL_COMPLETED_G'):
    # Only for the declared local integration test of serialized cells.
    # The author Colab always performs the complete calculation below.
    shutil.copytree(Path(os.environ['ALEM_BETA_LOCAL_COMPLETED_G']),OUTPUT)
    print('Integração local: reutiliza cálculo G local já concluído; não é uma nova simulação nem execução do autor no Colab.')
else:
    script('run_lab03g.py','--x-archive',X_ARCHIVE,'--output',OUTPUT,'--stage','all')


In [ ]:
script('verify_lab03g.py','--x-archive',X_ARCHIVE,'--output',OUTPUT)
script('plot_lab03g.py','--output',OUTPUT)
reference=ROOT/'results/tables/lab03g_pos_parecer_v3_20261024'
compared=0
for name in ['observed_aggregate.csv.gz','temporal_asset_points.csv','calibration_replications.csv.gz','calibration_cases.csv','calibration_families.csv']:
    observed=pd.read_csv(OUTPUT/name,float_precision='round_trip');expected=pd.read_csv(reference/name,float_precision='round_trip')
    pd.testing.assert_frame_equal(observed,expected,check_exact=False,atol=1e-10,rtol=1e-9);compared+=observed.size
with np.load(OUTPUT/'observed_draws.npz') as actual,np.load(reference/'observed_draws.npz') as expected:
    assert actual.files==expected.files
    for key in actual.files:np.testing.assert_allclose(actual[key],expected[key],atol=1e-10,rtol=1e-9);compared+=actual[key].size
for name,expected in SOURCE_PINS.items():assert digest(ROOT/name)==expected,name
record={'passed':True,'environment':'local_integration_reuses_completed_MC' if LOCAL_MODE else 'Colab','source_commit':SOURCE_COMMIT,'source_files':len(SOURCE_PINS),'compared_values':compared,'atol':1e-10,'rtol':1e-9,'X_sha256':digest(X_ARCHIVE),'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__}
(OUTPUT/'notebook_comparison.json').write_text(json.dumps(record,indent=2)+'\n')
print(json.dumps(record,indent=2))


In [ ]:
aggregate=pd.read_csv(OUTPUT/'observed_aggregate.csv.gz')
display(aggregate[aggregate.family.eq('temporal4')&aggregate.block.eq(21)][['profile','statistic','estimate','ci_low','ci_high','p_adjusted']])
calibration=pd.read_csv(OUTPUT/'calibration_families.csv');display(calibration[calibration.adjusted])
coverage=pd.read_csv(OUTPUT/'calibration_cases.csv');display(coverage[coverage.event.eq('coverage')].groupby(['mechanism','family']).rate.agg(['min','max']))
for name in ['lab03g_temporal','lab03g_calibracao']:display(Image(filename=str(OUTPUT/'figures'/(name+'.png'))))


## Interpretação e salvamento

Os valores-p são aproximações centradas, e os ICs básicos são marginais. A calibração expõe subcobertura: não interpretar o rótulo 95% como cobertura demonstrada. Controle sob nulidade global simulada não demonstra controle forte sob todas as alternativas ou leis financeiras. Os modelos dinâmicos/semanais e outros comprimentos ainda não receberam essa calibração.

Mudança média agregada não é mudança de cada ativo; ausência de rejeição não prova simetria, equivalência ou persistência. Retirar março ou um semestre muda o alvo temporal e não identifica uma causa. Auditoria econômica global, sobrevivência e dados disponíveis em cada época continuam limitando o alcance. Representação/raciocínio permanece uma proposta não avaliada.

O ZIP final preserva fontes, resultados, gráficos e hashes, sem a matriz privada de retornos. Ele será salvo em `alem_do_beta/lab03g_resultados`. **Salve também o notebook executado no Drive e envie o link da cópia.**


In [ ]:
export=RUN/('resultados_lab03g_'+STAMP+'.zip')
inventory={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for base,prefix in [(ROOT,'runtime'),(OUTPUT,'outputs')]:
        for path in sorted(base.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                name=prefix+'/'+str(path.relative_to(base));z.write(path,name);inventory[name]=digest(path)
    z.writestr('manifesto_exports.json',json.dumps(inventory,indent=2))
print('Export:',export.name,'| bytes:',export.stat().st_size,'| SHA256:',digest(export))
if not LOCAL_MODE:
    folder=Path('/content/drive/MyDrive/alem_do_beta/lab03g_resultados');folder.mkdir(parents=True,exist_ok=True)
    destination=folder/export.name;shutil.copy2(export,destination);assert digest(destination)==digest(export)
    print('SALVO NO DRIVE:',destination)
else:print('Integração local concluída. Não é execução Colab do autor.')
